[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap04/cap04.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

🚧 **Em construção!**

Esta seção está sendo preparada e será disponibilizada em breve com exercícios práticos, desafios de programação e exemplos.

Esta lista transforma os conceitos do Capítulo 4 em uma trilha prática de segmentação e morfologia matemática. Os EPs começam com limiarização e avançam até rotulação e descritores de componentes, sempre com matrizes pequenas para que cada pixel possa ser conferido à mão.

::: {.callout-important}
### Regra comum dos EPs morfológicos {.unnumbered}

Nas operações com vizinhança, **não faça padding**. Para cada pixel, avalie apenas as posições do elemento estruturante que caem dentro do domínio da imagem. Essa é a mesma ideia das implementações didáticas em `morph.py`, como `mm.dil0`, `mm.ero0`, `mm.dil1` e `mm.label0`: a vizinhança é recortada pelo domínio válido da imagem.
:::


### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de  **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [18]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.2 | TestSuite: 1.1.0


#### Executando os Testes {.unnumbered}
Para rodar os testes, execute `TestSuite("EP04_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como string numa variável `codigo`:

```python
codigo = """
from morph import mm
# ... seu código aqui ...
"""
TestSuite("EP04_01").run_code(codigo)
```

### EP04_01 🎚️ Limiarização Global por Limiar Fixo

Em **scanners de documentos** e em **sistemas de leitura de código de barras**, a primeira etapa do processamento é sempre separar o que é "objeto" (tinta, texto, barras) do que é "fundo" (papel, embalagem). A **limiarização global** faz exatamente isso: compara cada pixel a um único limiar $T$ e decide, em tempo real, se ele pertence à classe clara ou à classe escura. É o operador de segmentação mais simples — e mesmo assim, está por trás de boa parte dos *pipelines* industriais de inspeção visual.
Ver na @fig-04-sim-limiar uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Limiar:** Ler o inteiro $T$ (limiar de decisão).
3. **Dados:** Ler os valores inteiros da matriz original linha a linha.
4. **Mapeamento:** Para cada pixel $p$, calcular o novo valor pela equação:

$$
p' =
\begin{cases}
255, & \text{se } p > T \\
0, & \text{se } p \le T
\end{cases}
$$
5. **Saída:** Exibir a matriz binarizada com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Binarização:** A saída contém **apenas** os valores $0$ ou $255$.
* **Comparação estrita:** O critério usa $> T$ (pixels iguais a $T$ tornam-se fundo).
* **Tipo:** O resultado final deve ser inteiro.
* **Observação:** Este EP segue a convenção da OpenCV (`cv2.THRESL_BINARY`): apenas pixels com valor **maior que** $T$ tornam-se brancos (`255`); pixels com valor **igual a** $T$ permanecem pretos (`0`).

#### 🧠 Fundamentação Teórica

| Parâmetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$T$ pequeno** | Inteiro | A maioria dos pixels torna-se branca |
| **$T$ grande**  | Inteiro | A maioria dos pixels torna-se preta |
| **$T$ bem escolhido** | Inteiro | Separa nitidamente objeto e fundo |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $T$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**

* Matriz binarizada em $L$ linhas e $C$ colunas, valores $0$ ou $255$ separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 0 255<br>255 0 255 0 | $T=100$: apenas pixels com valor maior que 100 tornam-se brancos; <br>por isso, 99 e 100 tornam-se pretos. |
| 1<br>3<br>0<br>0 50 255 | 0 255 255 | $T=0$: apenas pixels com valor estritamente maior que 0 tornam-se brancos.|




In [19]:
#| label: fig-04-sim-limiar
#| fig-cap: "Simulador: Limiarização Global por Limiar Fixo"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0401b" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Limiarização Global</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🎚️ p' = (p > T) ? 255 : 0</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">👆 Clique numa célula de <b>Entrada Original</b> para escurecer o pixel (e clique direito para clarear)</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">T (Limiar)</label>
        <span id="ep0401b_vl_t" style="font-family:monospace;font-weight:bold;color:#2980b9;">128</span>
      </div>
      <input id="ep0401b_sl_t" style="width:100%;accent-color:#2980b9;" max="255" min="0" step="1" type="range" value="128">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original (clicável)</p>
        <div id="ep0401b_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0401b_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado (p')</p>
        <div id="ep0401b_grid_new" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0401b_btn_reset" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Resetar T</button>
      </div>
    </div>
    <div id="ep0401b_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0401bReady === '1') return;
    root.dataset.ep0401bReady = '1';

    var slT = root.querySelector('#ep0401b_sl_t');
    var vlT = root.querySelector('#ep0401b_vl_t');
    var gO  = root.querySelector('#ep0401b_grid_orig');
    var gN  = root.querySelector('#ep0401b_grid_new');
    var db  = root.querySelector('#ep0401b_debug');
    var btnNew   = root.querySelector('#ep0401b_btn_new');
    var btnReset = root.querySelector('#ep0401b_btn_reset');
    if(!slT || !gO || !gN || !db || !btnNew || !btnReset) return;

    var pixels = [];

    function generate(){ pixels = []; for(var i=0;i<16;i++) pixels.push(Math.floor(Math.random()*256)); }

    function renderOrig(){
      gO.innerHTML='';
      pixels.forEach(function(p, idx){
        var c = document.createElement('div');
        c.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;cursor:pointer;';
        c.style.background='rgb('+p+','+p+','+p+')';
        c.style.color = p>128 ? 'black' : 'white';
        c.innerText = p;
        c.title = 'clique esquerdo: -30 | clique direito: +30';
        c.onclick = function(){ pixels[idx] = Math.max(0, pixels[idx]-30); render(); };
        c.oncontextmenu = function(e){ e.preventDefault(); pixels[idx] = Math.min(255, pixels[idx]+30); render(); };
        gO.appendChild(c);
      });
    }

    function render(){
      var T = parseInt(slT.value, 10);
      vlT.innerText = T;
      db.innerHTML = 'Fórmula aplicada: <b>(p > '+T+') ? 255 : 0</b>';
      renderOrig();
      gN.innerHTML = '';
      pixels.forEach(function(p){
        var res = (p>T) ? 255 : 0;
        var c = document.createElement('div');
        c.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
        c.style.background='rgb('+res+','+res+','+res+')';
        c.style.color = res>128 ? 'black' : 'white';
        c.innerText = res;
        gN.appendChild(c);
      });
    }

    slT.oninput = render;
    btnNew.onclick = function(){ generate(); render(); };
    btnReset.onclick = function(){ slT.value = 128; render(); };

    generate();
    render();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0401b"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [20]:
%%writefile EP04_01.py
# Código Python

Writing EP04_01.py


In [21]:
TestSuite("EP04_01.py").run()

### EP04_02 📊 Limiarização Automática de Otsu

Escolher manualmente o limiar $T$ funciona quando a iluminação é estável, mas em **microscopia digital** e em **inspeção de lâminas de sangue**, cada amostra tem um contraste diferente — um limiar fixo falharia de imagem para imagem. O **método de Otsu** resolve isso encontrando, sozinho, o limiar que **maximiza a separação estatística** entre as duas classes de pixels, tornando a segmentação automática e adaptativa.
Ver na @fig-04-sim-otsu uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler os valores inteiros da matriz original linha a linha.
3. **Histograma:** Construir o histograma $h[i]$, $i=0,\dots,255$, contando quantos pixels têm valor $i$.
4. **Busca do limiar:** Para cada candidato $T$ de $1$ a $255$, calcular a **variância entre classes**:
$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$
onde $n_0,n_1$ são as quantidades de pixels com valor $<T$ e $\geq T$, $m_0,m_1$ são suas médias, e $N=L\times C$.

5. **Escolha:** O limiar ótimo $T^*$ é o que maximiza $\sigma_B^2(T)$ (em caso de empate, manter o **primeiro** encontrado).
6. Aplicação: Binarizar a imagem usando T*, aplicando:
$$
p' =
\begin{cases}
255, & \text{se } p > T^* \\
0, & \text{se } p \le T^*
\end{cases}
$$

#### 📌 Restrições Computacionais

* **Candidatos válidos:** Ignorar $T$ que deixe $n_0=0$ ou $n_1=0$ (classe vazia).
* **Empate:** Sempre manter o **primeiro** $T$ que atingiu o valor máximo de $\sigma_B^2$.
* **Tipo:** $T^*$ e a matriz de saída devem ser inteiros.
* **Convenção OpenCV:** A binarização segue `cv2.THRESL_BINARY`; pixels com valor exatamente igual a $T^*$ tornam-se pretos.


#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto |
|----------|-------------|---------|
| **$\sigma_B^2(T)$ alta** | Classes bem separadas em $T$ | $T$ é um bom candidato a limiar |
| **Histograma bimodal** | Dois "morros" distintos | Otsu encontra o vale entre eles |
| **Histograma unimodal** | Um único "morro" | Otsu ainda escolhe *algum* $T$, mas a segmentação é pouco confiável |


#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**

* Matriz binarizada em $L$ linhas e $C$ colunas, valores $0$ ou $255$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Histograma bimodal claro: 12 e 200 |
| 1<br>2<br>10 250 | 0 250 | Apenas dois valores: $T^*$ fica no maior |


In [22]:
#| label: fig-04-sim-otsu
#| fig-cap: "Simulador: Limiarização Automática de Otsu"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0402" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Otsu Automático</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">📊 T* = argmax σ²ᴮ(T)</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">👆 Clique esquerdo escurece, clique direito clarece o pixel — veja o T* mudar sozinho</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:20px;">
      <svg id="ep0402_hist" height="120" style="width:100%;display:block;" viewBox="0 0 280 120"></svg>
      <p id="ep0402_info" style="text-align:center;font-size:12px;color:#555;margin-top:8px;">T* = -</p>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original (clicável)</p>
        <div id="ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado (Otsu)</p>
        <div id="ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div style="text-align:center;margin-top:15px;">
      <button id="ep0402_btn_new" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem (gera dois grupos)</button>
    </div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0402Ready === '1') return;
    root.dataset.ep0402Ready = '1';

    var gO = root.querySelector('#ep0402_grid_orig');
    var gN = root.querySelector('#ep0402_grid_new');
    var info = root.querySelector('#ep0402_info');
    var svg = root.querySelector('#ep0402_hist');
    var btnNew = root.querySelector('#ep0402_btn_new');
    if(!gO || !gN || !info || !svg || !btnNew) return;

    var pixels = [];

    function generate(){
      var c1 = 20 + Math.floor(Math.random()*40);
      var c2 = 180 + Math.floor(Math.random()*60);
      pixels = [];
      for (var i=0;i<16;i++){
        var base = (Math.random()<0.5) ? c1 : c2;
        pixels.push(Math.max(0, Math.min(255, base + Math.floor(Math.random()*16-8))));
      }
    }

    function otsu(pix){
      var hist = new Array(256).fill(0);
      pix.forEach(function(p){ hist[p]++; });
      var N = pix.length, bestVar = -1, bestT = 0;
      var total = pix.reduce(function(a,b){return a+b;}, 0);
      for (var T=1; T<256; T++){
        var n0=0, s0=0;
        for (var i=0;i<T;i++){ n0+=hist[i]; s0+=i*hist[i]; }
        var n1 = N-n0, s1 = total-s0;
        if (n0===0 || n1===0) continue;
        var m0=s0/n0, m1=s1/n1;
        var v=(n0*n1)*(m0-m1)*(m0-m1)/(N*N);
        if (v>bestVar){ bestVar=v; bestT=T; }
      }
      return bestT;
    }

    function renderOrig(T){
      gO.innerHTML='';
      pixels.forEach(function(p, idx){
        var c = document.createElement('div');
        c.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;cursor:pointer;';
        c.style.background='rgb('+p+','+p+','+p+')';
        c.style.color = p>128 ? 'black' : 'white';
        c.innerText = p;
        c.title = 'clique esquerdo: -25 | clique direito: +25';
        c.onclick = function(){ pixels[idx] = Math.max(0, pixels[idx]-25); render(); };
        c.oncontextmenu = function(e){ e.preventDefault(); pixels[idx] = Math.min(255, pixels[idx]+25); render(); };
        gO.appendChild(c);
      });
    }

    function render(){
      var T = otsu(pixels);
      info.innerHTML = 'T* encontrado = <b>'+T+'</b>';
      renderOrig(T);
      gN.innerHTML='';
      var hist = new Array(256).fill(0);
      pixels.forEach(function(p){ hist[p]++; });
      var maxH = Math.max.apply(null, hist);
      var bars = '';
      for (var i=0;i<256;i+=4){
        var h = (hist[i]/(maxH||1))*100;
        var col = (i>=T) ? '#2980b9' : '#888';
        bars += '<rect x="'+(i*280/256)+'" y="'+(115-h)+'" width="4" height="'+h+'" fill="'+col+'"></rect>';
      }
      bars += '<line x1="'+(T*280/256)+'" y1="0" x2="'+(T*280/256)+'" y2="115" stroke="#e74c3c" stroke-width="2" stroke-dasharray="3,2"></line>';
      svg.innerHTML = bars;
      pixels.forEach(function(p){
        var res = (p>T) ? 255 : 0;
        var c = document.createElement('div');
        c.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
        c.style.background='rgb('+res+','+res+','+res+')';
        c.style.color = res>128 ? 'black' : 'white';
        c.innerText = res;
        gN.appendChild(c);
      });
    }

    btnNew.onclick = function(){ generate(); render(); };

    generate();
    render();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0402"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [23]:
%%writefile EP04_02.py
# Código Python

Writing EP04_02.py


In [24]:
TestSuite("EP04_02.py").run()

### EP04_03 🌱 Dilatação Binária Plana (mm.dil0)

Em **microscopia de partículas** e em **OCR de placas de carro desgastadas**, traços finos ou descontínuos precisam ser "engrossados" para que o reconhecimento funcione. A **dilatação morfológica** faz exatamente isso: expande regiões claras usando um elemento estruturante $B$ — a mesma operação implementada em `morph.py` como `mm.dil0(f, B)`, usada quando $B$ é **plano** (sem pesos, só $0$/$1$).
Ver na @fig-04-sim-dil0 uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original), linha a linha.
5. **Reflexão:** Construir $B_{ref}$, a versão de $B$ refletida em $180°$ (linhas e colunas invertidas) — exatamente como faz `mm.dil0` internamente.
6. **Vizinhança sem padding:** Para cada pixel $(y,x)$, percorrer as posições $(by,bx)$ de $B_{ref}$ centradas em $(y,x)$, usando o deslocamento
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Descartar** todo $(v_y,v_x)$ fora de $[0,L)\times[0,C)$ — **não preencher com zeros**.
7. **Mapeamento:** Calcular cada pixel de saída como o **máximo** entre $f(y,x)$ e todos os $f(v_y,v_x)$ válidos cuja posição correspondente em $B_{ref}$ vale $1$:
$$
g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{válido}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)
$$
8. **Saída:** Exibir a matriz $g$ com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Sem padding:** Jamais inventar vizinhos fora da imagem; usar apenas os que existem de fato.
* **Reflexão obrigatória:** $B$ deve ser refletido antes de aplicado (é o que diferencia `mm.dil0` de uma simples busca por máximo).
* **Robustez de borda:** Se nenhuma posição válida de $B_{ref}=1$ cair dentro do domínio para um dado pixel, ele **mantém seu valor original**.

#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Dilatação** | $g \geq f$ sempre (extensiva) | Regiões claras crescem, buracos escuros encolhem |
| **$B$ maior** | Vizinhança mais ampla | Crescimento mais agressivo |
| **Reflexão de $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Garante a definição formal de Minkowski da dilatação |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $L_B$.
* Linha 4: Inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.

**Saída:**

* Matriz $g$ em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | $B$ em cruz simétrico: ponto isolado se expande em cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | $B$ horizontal: cada pixel "puxa" o máximo dos vizinhos da linha |


In [25]:
#| label: fig-04-sim-dil0
#| fig-cap: "Simulador: Dilatação Binária Plana (mm.dil0)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0403" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Dilatação Plana (mm.dil0)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🌱 g = f ⊕ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <p style="font-size:11px;font-weight:bold;color:#16a085;margin-bottom:10px;text-align:center;">Elemento Estruturante B (clique para alternar 0/1)</p>
      <div id="ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3,38px);gap:4px;justify-content:center;margin-bottom:10px;"></div>
      <div style="display:flex;gap:8px;justify-content:center;">
        <button id="ep0403_btn_cross" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">➕ Cruz</button>
        <button id="ep0403_btn_box" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⬛ Caixa</button>
        <button id="ep0403_btn_diag" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⤫ Diagonal</button>
      </div>
    </div>
    <p style="font-size:11px;color:#777;margin-bottom:10px;text-align:center;">👆 Clique nas células de <b>f</b> para acender/apagar pixels</p>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Imagem Original (f)</p>
        <div id="ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0403_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Dilatada (g)</p>
        <div id="ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0403_debug" style="margin-top:20px;background:#e8f5e9;border-radius:8px;padding:10px;border:1px solid #c8e6c9;font-family:monospace;font-size:11px;color:#2e7d32;text-align:center;">g(y,x) = max sobre vizinhos válidos de B refletido</div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0403Ready === '1') return;
    root.dataset.ep0403Ready = '1';

    var gB = root.querySelector('#ep0403_grid_B');
    var gO = root.querySelector('#ep0403_grid_orig');
    var gN = root.querySelector('#ep0403_grid_new');
    var db = root.querySelector('#ep0403_debug');
    var btnNew  = root.querySelector('#ep0403_btn_new');
    var btnCross = root.querySelector('#ep0403_btn_cross');
    var btnBox   = root.querySelector('#ep0403_btn_box');
    var btnDiag  = root.querySelector('#ep0403_btn_diag');
    if(!gB || !gO || !gN || !db || !btnNew || !btnCross || !btnBox || !btnDiag) return;

    var B = [[0,1,0],[1,1,1],[0,1,0]];
    var L=5, C=5, pixels=[];

    function generate(){
      pixels=[];
      for(var y=0;y<L;y++){ var row=[]; for(var x=0;x<C;x++) row.push((Math.random()<0.78)?0:1); pixels.push(row); }
    }
    function reflect(M){ var n=M.length, out=[]; for(var i=n-1;i>=0;i--) out.push(M[i].slice().reverse()); return out; }
    function dilate(f,Bm){
      var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var Bref=reflect(Bm);
      var g=[]; for(var y=0;y<L;y++) g.push(f[y].slice());
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) for(var by=0;by<HB;by++) for(var bx=0;bx<WB;bx++){
        if(Bref[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0 && vy<L && vx>=0 && vx<C && f[vy][vx]>g[y][x]) g[y][x]=f[vy][vx];
      }
      return g;
    }
    function renderB(){
      gB.innerHTML='';
      for(var by=0;by<3;by++) for(var bx=0;bx<3;bx++){
        (function(byy,bxx){
          var c=document.createElement('div');
          c.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:13px;font-weight:bold;border-radius:4px;cursor:pointer;border:1px solid #ccc;';
          c.style.background = B[byy][bxx] ? '#16a085' : '#f5f5f5';
          c.style.color = B[byy][bxx] ? 'white' : '#999';
          c.innerText = B[byy][bxx];
          c.onclick=function(){ B[byy][bxx]=1-B[byy][bxx]; renderB(); render(); };
          gB.appendChild(c);
        })(by,bx);
      }
    }
    function render(){
      var g = dilate(pixels,B);
      gO.innerHTML=''; gN.innerHTML='';
      for(var y=0;y<L;y++) for(var x=0;x<C;x++){
        (function(yy,xx){
          var p = pixels[yy][xx] ? 255 : 30;
          var cO=document.createElement('div');
          cO.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;cursor:pointer;';
          cO.style.background='rgb('+p+','+p+','+p+')'; cO.style.color = p>128 ? 'black' : 'white';
          cO.innerText = pixels[yy][xx];
          cO.onclick=function(){ pixels[yy][xx]=1-pixels[yy][xx]; render(); };
          gO.appendChild(cO);
        })(y,x);
        var r = g[y][x] ? 255 : 30;
        var cN=document.createElement('div');
        cN.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #ddd;';
        cN.style.background='rgb('+r+','+r+','+r+')'; cN.style.color = r>128 ? 'black' : 'white';
        cN.innerText = g[y][x];
        gN.appendChild(cN);
      }
    }

    btnNew.onclick   = function(){ generate(); render(); };
    btnCross.onclick = function(){ B=[[0,1,0],[1,1,1],[0,1,0]]; renderB(); render(); };
    btnBox.onclick   = function(){ B=[[1,1,1],[1,1,1],[1,1,1]]; renderB(); render(); };
    btnDiag.onclick  = function(){ B=[[1,0,1],[0,1,0],[1,0,1]]; renderB(); render(); };

    generate();
    renderB();
    render();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0403"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [26]:
%%writefile EP04_03.py
# Código Python

Writing EP04_03.py


In [27]:
TestSuite("EP04_03.py").run()


### EP04_04 🪨 Erosão Binária Plana (mm.ero0)

Se a dilatação engrossa, a **erosão** afina. Em **sistemas de contagem de células**, ela é usada para **separar células que se tocam**: ao "comer" as bordas de cada região, conexões finas entre objetos desaparecem antes mesmo de qualquer contagem ser feita. Em `morph.py`, essa é a operação `mm.ero0(f, B)` — a **dual** exata da dilatação, e a única das duas que **não** reflete o elemento estruturante.
Ver na @fig-04-sim-ero0 uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original), linha a linha.
5. **Vizinhança sem padding (sem reflexão!):** Para cada pixel $(y,x)$, percorrer as posições $(by,bx)$ de $B$ **na ordem original** (sem refletir), usando o mesmo deslocamento do EP04_03:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$

**Descartar** todo $(v_y,v_x)$ fora de $[0,L)\times[0,C)$.
6. **Mapeamento:** Calcular cada pixel de saída como o **mínimo** entre $f(y,x)$ e todos os $f(v_y,v_x)$ válidos cuja posição correspondente em $B$ vale $1$:
$$
g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{válido}\\ B(by,bx)=1}} f(v_y,v_x)\Big)
$$
7. **Saída:** Exibir a matriz $g$ com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Sem reflexão:** Diferente da dilatação, $B$ é usado **exatamente como lido** — refletir aqui seria um erro conceitual grave.
* **Sem padding:** Vizinhos fora da imagem são simplesmente ignorados, nunca tratados como $0$.
* **Robustez de borda:** Se nenhuma posição válida de $B=1$ cair dentro do domínio, o pixel mantém seu valor original.

#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Erosão** | $g \leq f$ sempre (anti-extensiva) | Regiões claras encolhem, ruído pontual desaparece |
| **Dualidade** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | Erosão e dilatação são "espelhos" matemáticos |
| **$B$ maior** | Erosão mais agressiva | Objetos finos somem completamente |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $L_B$.
* Linha 4: Inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.

**Saída:**

* Matriz $g$ em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | O "buraco" central (0) se propaga em cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | $B$ horizontal: cada pixel "puxa" o mínimo dos vizinhos da linha |


In [28]:
#| label: fig-04-sim-ero0
#| fig-cap: "Simulador: Erosão Binária Plana (mm.ero0)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0404" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Erosão Plana (mm.ero0)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🪨 g = f ⊖ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <p style="font-size:11px;font-weight:bold;color:#c0392b;margin-bottom:10px;text-align:center;">Elemento Estruturante B (clique para alternar 0/1)</p>
      <div id="ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3,38px);gap:4px;justify-content:center;margin-bottom:10px;"></div>
      <div style="display:flex;gap:8px;justify-content:center;">
        <button id="ep0404_btn_cross" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">➕ Cruz</button>
        <button id="ep0404_btn_box" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⬛ Caixa</button>
        <button id="ep0404_btn_diag" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⤫ Diagonal</button>
      </div>
    </div>
    <p style="font-size:11px;color:#777;margin-bottom:10px;text-align:center;">👆 Clique nas células de <b>f</b> para acender/apagar pixels</p>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Imagem Original (f)</p>
        <div id="ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0404_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Erodida (g)</p>
        <div id="ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0404_debug" style="margin-top:20px;background:#fdecea;border-radius:8px;padding:10px;border:1px solid #f5c6cb;font-family:monospace;font-size:11px;color:#a93226;text-align:center;">g(y,x) = min sobre vizinhos válidos de B (sem refletir)</div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0404Ready === '1') return;
    root.dataset.ep0404Ready = '1';

    var gB = root.querySelector('#ep0404_grid_B');
    var gO = root.querySelector('#ep0404_grid_orig');
    var gN = root.querySelector('#ep0404_grid_new');
    var btnNew  = root.querySelector('#ep0404_btn_new');
    var btnCross = root.querySelector('#ep0404_btn_cross');
    var btnBox   = root.querySelector('#ep0404_btn_box');
    var btnDiag  = root.querySelector('#ep0404_btn_diag');
    if(!gB || !gO || !gN || !btnNew || !btnCross || !btnBox || !btnDiag) return;

    var B = [[0,1,0],[1,1,1],[0,1,0]];
    var L=5, C=5, pixels=[];

    function generate(){
      pixels=[];
      for(var y=0;y<L;y++){ var row=[]; for(var x=0;x<C;x++) row.push((Math.random()<0.78)?1:0); pixels.push(row); }
    }
    function erode(f,Bm){
      var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var g=[]; for(var y=0;y<L;y++) g.push(f[y].slice());
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) for(var by=0;by<HB;by++) for(var bx=0;bx<WB;bx++){
        if(Bm[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0 && vy<L && vx>=0 && vx<C && f[vy][vx]<g[y][x]) g[y][x]=f[vy][vx];
      }
      return g;
    }
    function renderB(){
      gB.innerHTML='';
      for(var by=0;by<3;by++) for(var bx=0;bx<3;bx++){
        (function(byy,bxx){
          var c=document.createElement('div');
          c.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:13px;font-weight:bold;border-radius:4px;cursor:pointer;border:1px solid #ccc;';
          c.style.background = B[byy][bxx] ? '#c0392b' : '#f5f5f5';
          c.style.color = B[byy][bxx] ? 'white' : '#999';
          c.innerText = B[byy][bxx];
          c.onclick=function(){ B[byy][bxx]=1-B[byy][bxx]; renderB(); render(); };
          gB.appendChild(c);
        })(by,bx);
      }
    }
    function render(){
      var g = erode(pixels,B);
      gO.innerHTML=''; gN.innerHTML='';
      for(var y=0;y<L;y++) for(var x=0;x<C;x++){
        (function(yy,xx){
          var p = pixels[yy][xx] ? 255 : 30;
          var cO=document.createElement('div');
          cO.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;cursor:pointer;';
          cO.style.background='rgb('+p+','+p+','+p+')'; cO.style.color = p>128 ? 'black' : 'white';
          cO.innerText = pixels[yy][xx];
          cO.onclick=function(){ pixels[yy][xx]=1-pixels[yy][xx]; render(); };
          gO.appendChild(cO);
        })(y,x);
        var r = g[y][x] ? 255 : 30;
        var cN=document.createElement('div');
        cN.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #ddd;';
        cN.style.background='rgb('+r+','+r+','+r+')'; cN.style.color = r>128 ? 'black' : 'white';
        cN.innerText = g[y][x];
        gN.appendChild(cN);
      }
    }

    btnNew.onclick   = function(){ generate(); render(); };
    btnCross.onclick = function(){ B=[[0,1,0],[1,1,1],[0,1,0]]; renderB(); render(); };
    btnBox.onclick   = function(){ B=[[1,1,1],[1,1,1],[1,1,1]]; renderB(); render(); };
    btnDiag.onclick  = function(){ B=[[1,0,1],[0,1,0],[1,0,1]]; renderB(); render(); };

    generate();
    renderB();
    render();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0404"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [29]:
%%writefile EP04_04.py
# Código Python

Writing EP04_04.py


In [30]:
TestSuite("EP04_04.py").run()

### EP04_05 🧹 Abertura Morfológica (Remoção de Ruído)

Imagens capturadas por **sensores de baixo custo**, como os de drones agrícolas, costumam vir salpicadas de pequenos pontos de ruído — pixels isolados que não representam nada real. Aplicar erosão seguida de dilatação com o **mesmo** elemento estruturante produz a **abertura**: ela "limpa" pontos e protuberâncias finas, mas devolve ao objeto principal praticamente seu tamanho original. É a combinação clássica usada em **pré-processamento de imagens de satélite** antes de qualquer contagem de área plantada.
Ver na @fig-04-sim-abertura uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Erosão:** Calcular $e = f \ominus B$, usando exatamente o algoritmo do EP04_04 (sem refletir $B$, sem padding).
6. **Dilatação:** Calcular $g = e \oplus B$, usando exatamente o algoritmo do EP04_03 (refletindo $B$, sem padding) — mas agora aplicado sobre $e$, não sobre $f$.
7. **Saída:** Exibir a matriz resultante $g$ (a **abertura** de $f$ por $B$) com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Ordem fixa:** É **sempre** erosão primeiro, depois dilatação — a ordem inversa define outro operador (fechamento, do próximo EP).
* **Mesmo $B$:** O elemento estruturante usado na erosão e na dilatação deve ser idêntico.
* **Sem padding em nenhuma das duas etapas.**

#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Anti-extensividade** | $g \subseteq f$ sempre | A abertura nunca cria pixel novo, só remove |
| **Idempotência** | $\text{abertura}(\text{abertura}(f)) = \text{abertura}(f)$ | Aplicar de novo não muda mais nada |
| **Pontos isolados** | Menores que $B$ | São completamente eliminados |
| **Núcleo do objeto** | Maior que $B$ | É recuperado quase intacto pela dilatação final |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $L_B$.
* Linha 4: Inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros ($0$ ou $1$) da matriz $f$.

**Saída:**

* Matriz resultante em $L$ linhas e $C$ colunas, valores $0$ ou $1$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Pontos isolados e a protuberância fina desaparecem; o quadrado central sobrevive |


In [31]:
#| label: fig-04-sim-abertura
#| fig-cap: "Simulador: Abertura Morfológica (erosão + dilatação)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0405" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Abertura Morfológica</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🧹 g = (f ⊖ B) ⊕ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">👆 Clique nas células de <b>f original</b> para acender/apagar pixels (crie seu próprio ruído!)</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:20px;text-align:center;">
      <label style="font-size:12px;font-weight:bold;color:#8e44ad;">Tamanho de B (caixa n×n)</label><br>
      <input id="ep0405_sl_n" style="width:60%;accent-color:#8e44ad;margin-top:8px;" max="5" min="3" step="2" type="range" value="3">
      <span id="ep0405_vl_n" style="font-family:monospace;font-weight:bold;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original (clicável)</p>
        <div id="ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7,28px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">e = f ⊖ B</p>
        <div id="ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7,28px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">g = e ⊕ B</p>
        <div id="ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7,28px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div style="text-align:center;margin-top:15px;">
      <button id="ep0405_btn_new" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem (com ruído)</button>
      <button id="ep0405_btn_clear" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;margin-left:6px;">🧹 Limpar Tudo</button>
    </div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0405Ready === '1') return;
    root.dataset.ep0405Ready = '1';

    var slN = root.querySelector('#ep0405_sl_n');
    var vlN = root.querySelector('#ep0405_vl_n');
    var gF = root.querySelector('#ep0405_grid_f');
    var gE = root.querySelector('#ep0405_grid_e');
    var gG = root.querySelector('#ep0405_grid_g');
    var btnNew = root.querySelector('#ep0405_btn_new');
    var btnClear = root.querySelector('#ep0405_btn_clear');
    if(!slN || !gF || !gE || !gG || !btnNew || !btnClear) return;

    var L=7, C=7, f=[];

    function generate(){
      f=[]; for(var y=0;y<L;y++){ var row=[]; for(var x=0;x<C;x++) row.push(0); f.push(row); }
      for(var y=2;y<5;y++) for(var x=2;x<5;x++) f[y][x]=1;
      for(var k=0;k<3;k++){
        var ry=Math.floor(Math.random()*L), rx=Math.floor(Math.random()*C);
        if(f[ry][rx]===0 && (ry<1||ry>5||rx<1||rx>5)) f[ry][rx]=1;
      }
    }
    function clearAll(){ for(var y=0;y<L;y++) for(var x=0;x<C;x++) f[y][x]=0; }
    function box(n){ var B=[]; for(var i=0;i<n;i++){ var row=[]; for(var j=0;j<n;j++) row.push(1); B.push(row); } return B; }
    function reflect(M){ var n=M.length, out=[]; for(var i=n-1;i>=0;i--) out.push(M[i].slice().reverse()); return out; }
    function erode(img,Bm){
      var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var g=[]; for(var y=0;y<L;y++) g.push(img[y].slice());
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) for(var by=0;by<HB;by++) for(var bx=0;bx<WB;bx++){
        if(Bm[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0 && vy<L && vx>=0 && vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
      }
      return g;
    }
    function dilate(img,Bm){
      var Bref=reflect(Bm);
      var HB=Bref.length, WB=Bref[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var g=[]; for(var y=0;y<L;y++) g.push(img[y].slice());
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) for(var by=0;by<HB;by++) for(var bx=0;bx<WB;bx++){
        if(Bref[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0 && vy<L && vx>=0 && vx<C && img[vy][vx]>g[y][x]) g[y][x]=img[vy][vx];
      }
      return g;
    }
    function paintStatic(grid,img){
      grid.innerHTML='';
      for(var y=0;y<L;y++) for(var x=0;x<C;x++){
        var c=document.createElement('div');
        c.style.cssText='width:28px;height:28px;border-radius:3px;border:1px solid #ddd;';
        c.style.background = img[y][x]===1 ? '#8e44ad' : '#f7f3fb';
        grid.appendChild(c);
      }
    }
    function paintEditable(grid,img){
      grid.innerHTML='';
      for(var y=0;y<L;y++) for(var x=0;x<C;x++){
        (function(yy,xx){
          var c=document.createElement('div');
          c.style.cssText='width:28px;height:28px;border-radius:3px;border:1px solid #ccc;cursor:pointer;';
          c.style.background = img[yy][xx]===1 ? '#8e44ad' : '#f7f3fb';
          c.onclick=function(){ f[yy][xx]=1-f[yy][xx]; render(); };
          grid.appendChild(c);
        })(y,x);
      }
    }
    function render(){
      var n=parseInt(slN.value,10);
      vlN.innerText=n+'×'+n;
      var B=box(n);
      var e=erode(f,B);
      var g=dilate(e,B);
      paintEditable(gF,f); paintStatic(gE,e); paintStatic(gG,g);
    }

    slN.oninput = render;
    btnNew.onclick = function(){ generate(); render(); };
    btnClear.onclick = function(){ clearAll(); render(); };

    generate();
    render();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0405"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [32]:
%%writefile EP04_05.py
# Código Python

Writing EP04_05.py


In [33]:
TestSuite("EP04_05.py").run()

### EP04_06 🧩 Fechamento Morfológico (Preenchimento de Falhas)

Em **digitalização de impressões digitais**, sulcos da pele às vezes ficam interrompidos por sujeira ou ressecamento, criando pequenas falhas na curva contínua que deveria existir. O **fechamento** — dilatação seguida de erosão com o mesmo elemento estruturante — é o operador dual da abertura: ele **preenche buracos pequenos e reentrâncias estreitas**, sem alterar significativamente o contorno externo do objeto. É a etapa padrão antes de extrair o esqueleto de uma impressão digital.
Ver na @fig-04-sim-fechamento uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Dilatação:** Calcular $d = f \oplus B$, usando exatamente o algoritmo do EP04_03 (refletindo $B$, sem padding).
6. **Erosão:** Calcular $g = d \ominus B$, usando exatamente o algoritmo do EP04_04 (sem refletir $B$, sem padding) — agora aplicado sobre $d$, não sobre $f$.
7. **Saída:** Exibir a matriz resultante $g$ (o **fechamento** de $f$ por $B$) com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Ordem fixa:** É **sempre** dilatação primeiro, depois erosão — a ordem inversa é a abertura do EP04_05.
* **Mesmo $B$:** O elemento estruturante usado na dilatação e na erosão deve ser idêntico.
* **Sem padding em nenhuma das duas etapas.**

#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Extensividade** | $g \supseteq f$ sempre | O fechamento nunca remove pixel, só adiciona |
| **Idempotência** | $\text{fecha}(\text{fecha}(f)) = \text{fecha}(f)$ | Aplicar de novo não muda mais nada |
| **Buracos pequenos** | Menores que $B$ | São completamente preenchidos |
| **Dualidade** | $\text{fecha}(f) = \overline{\text{abre}(\bar f)}$ | É a abertura aplicada ao "negativo" da imagem |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $L_B$.
* Linha 4: Inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros ($0$ ou $1$) da matriz $f$.

**Saída:**

* Matriz resultante em $L$ linhas e $C$ colunas, valores $0$ ou $1$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | Os dois buracos internos não-adjacentes são totalmente preenchidos |


In [34]:
#| label: fig-04-sim-fechamento
#| fig-cap: "Simulador: Fechamento Morfológico (dilatação + erosão)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0406" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Fechamento Morfológico</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🧩 g = (f ⊕ B) ⊖ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">👆 Clique nas células de <b>f original</b> para acender/apagar pixels</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:20px;text-align:center;">
      <label style="font-size:12px;font-weight:bold;color:#2c7a7b;">Tamanho de B (caixa n×n)</label><br>
      <input id="ep0406_sl_n" style="width:60%;accent-color:#2c7a7b;margin-top:8px;" max="5" min="3" step="2" type="range" value="3">
      <span id="ep0406_vl_n" style="font-family:monospace;font-weight:bold;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original (clicável)</p>
        <div id="ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8,26px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">d = f ⊕ B</p>
        <div id="ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8,26px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">g = d ⊖ B</p>
        <div id="ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8,26px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div style="text-align:center;margin-top:15px;">
      <button id="ep0406_btn_new" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem (com buracos)</button>
      <button id="ep0406_btn_clear" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;margin-left:6px;">🧹 Limpar Tudo</button>
    </div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0406Ready === '1') return;
    root.dataset.ep0406Ready = '1';

    var slN = root.querySelector('#ep0406_sl_n');
    var vlN = root.querySelector('#ep0406_vl_n');
    var gF  = root.querySelector('#ep0406_grid_f');
    var gD  = root.querySelector('#ep0406_grid_d');
    var gG  = root.querySelector('#ep0406_grid_g');
    var btnNew   = root.querySelector('#ep0406_btn_new');
    var btnClear = root.querySelector('#ep0406_btn_clear');
    if(!slN || !gF || !gD || !gG || !btnNew || !btnClear) return;

    var L=8, C=8, f=[];

    function generate(){
      f = [];
      for(var y=0;y<L;y++){ var row=[]; for(var x=0;x<C;x++) row.push(0); f.push(row); }
      for(var y=1;y<7;y++) for(var x=2;x<6;x++) f[y][x]=1;
      f[3][3]=0; f[4][4]=0;
    }
    function clearAll(){
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) f[y][x]=0;
    }
    function box(n){
      var B=[]; for(var i=0;i<n;i++){ var row=[]; for(var j=0;j<n;j++) row.push(1); B.push(row); } return B;
    }
    function reflect(M){
      var n=M.length, out=[];
      for(var i=n-1;i>=0;i--) out.push(M[i].slice().reverse());
      return out;
    }
    function dilate(img, Bm){
      var Bref=reflect(Bm);
      var HB=Bref.length, WB=Bref[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var g=[]; for(var y=0;y<L;y++) g.push(img[y].slice());
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) for(var by=0;by<HB;by++) for(var bx=0;bx<WB;bx++){
        if(Bref[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0 && vy<L && vx>=0 && vx<C && img[vy][vx]>g[y][x]) g[y][x]=img[vy][vx];
      }
      return g;
    }
    function erode(img, Bm){
      var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var g=[]; for(var y=0;y<L;y++) g.push(img[y].slice());
      for(var y=0;y<L;y++) for(var x=0;x<C;x++) for(var by=0;by<HB;by++) for(var bx=0;bx<WB;bx++){
        if(Bm[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0 && vy<L && vx>=0 && vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
      }
      return g;
    }
    function paintStatic(grid, img){
      grid.innerHTML='';
      for(var y=0;y<L;y++) for(var x=0;x<C;x++){
        var c=document.createElement('div');
        var on=img[y][x]===1;
        c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ddd;';
        c.style.background = on ? '#2c7a7b' : '#eef7f7';
        grid.appendChild(c);
      }
    }
    function paintEditable(grid, img){
      grid.innerHTML='';
      for(var y=0;y<L;y++) for(var x=0;x<C;x++){
        (function(yy,xx){
          var c=document.createElement('div');
          var on=img[yy][xx]===1;
          c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ccc;cursor:pointer;';
          c.style.background = on ? '#2c7a7b' : '#eef7f7';
          c.title = 'linha '+yy+', coluna '+xx;
          c.onclick=function(){ f[yy][xx] = 1 - f[yy][xx]; render(); };
          grid.appendChild(c);
        })(y,x);
      }
    }
    function render(){
      var n=parseInt(slN.value, 10);
      vlN.innerText = n+'×'+n;
      var B=box(n);
      var d=dilate(f,B);
      var g=erode(d,B);
      paintEditable(gF, f);
      paintStatic(gD, d);
      paintStatic(gG, g);
    }

    slN.oninput = render;
    btnNew.onclick = function(){ generate(); render(); };
    btnClear.onclick = function(){ clearAll(); render(); };

    generate();
    render();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0406"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [35]:
%%writefile EP04_06.py
# Código Python

Writing EP04_06.py


In [36]:
TestSuite("EP04_06.py").run()

### EP04_07 ⛰️ Dilatação e Erosão com Pesos (mm.dil1 / mm.ero1)

Até agora, o elemento estruturante só dizia "este vizinho conta" ou "não conta" — mas em **modelos digitais de elevação** (usados em SIG e em planejamento de drenagem urbana), cada vizinho deveria ter um **peso diferente** dependendo da distância ou da direção do relevo. As versões **ponderadas** da dilatação e da erosão, implementadas em `morph.py` como `mm.dil1(f, b)` e `mm.ero1(f, b)`, somam (ou subtraem) o peso de cada vizinho antes de tomar o máximo (ou mínimo) — generalizando tudo o que foi feito nos EPs anteriores.
Ver na @fig-04-sim-pesos uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $b$:** Ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante ponderado.
3. **Pesos:** Ler a matriz $b$ de pesos **inteiros** (podem ser negativos, zero ou positivos), linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original), linha a linha.
5. **Vizinhança sem padding:** Para cada pixel $(y,x)$, percorrer **todas** as posições $(by,bx)$ de $b$ (não apenas onde valeria $1$ — aqui **todo** peso participa), usando o mesmo deslocamento dos EPs anteriores:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Descartar** todo $(v_y,v_x)$ fora de $[0,L)\times[0,C)$.
6. **Dilatação ponderada:** Calcular
$$
g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)
$$
7. **Erosão ponderada:** Calcular, **usando o mesmo $b$ e sem refletir**:
$$
g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)
$$
8. **Saída:** Exibir **primeiro** a matriz $g_{dil}$ completa, e **depois** a matriz $g_{ero}$ completa.

#### 📌 Restrições Computacionais

* **Nenhuma das duas reflete $b$** — a versão ponderada não usa reflexão, mesmo na dilatação (diferente de `mm.dil0`).
* **Todos os pesos participam:** Não existe aqui o filtro "$B=1$"; mesmo peso $0$ entra na conta.
* **Sem padding:** vizinhos fora da imagem são ignorados, nunca virtualmente preenchidos.
* **Tipo:** A saída pode conter valores negativos ou maiores que $255$ — **não** há *clipping* neste EP.
* **Dica:** Para remover mensagens de overflow ao ultrapassar limites do tipo uint8, incluir no início do código:
```python
import warnings
warnings.filterwarnings("ignore")
```

#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Peso positivo** | "Puxa" o valor do vizinho para cima na dilatação | Simula relevo que sobe naquela direção |
| **Peso negativo** | Reduz a contribuição do vizinho | Simula distância ou atenuação direcional |
| **Dualidade ponderada** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | A simetria entre as duas operações se mantém mesmo com pesos |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $L_B$.
* Linha 4: Inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros (podem ser negativos) da matriz $b$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.

**Saída:**

* Primeiro a matriz $g_{dil}$ em $L$ linhas e $C$ colunas.
* Em seguida a matriz $g_{ero}$ em $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Peso central $2$ acelera o crescimento na dilatação e o encolhimento na erosão |


In [37]:
#| label: fig-04-sim-pesos
#| fig-cap: "Simulador: Dilatação e Erosão com Pesos (mm.dil1 / mm.ero1)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0407" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pesos no Elemento Estruturante</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">⛰️ dil1 / ero1</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <p style="font-size:11px;font-weight:bold;color:#d35400;margin-bottom:10px;text-align:center;">Pesos b (clique e arraste o slider de cada célula)</p>
      <div id="ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3,56px);gap:6px;justify-content:center;"></div>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original</p>
        <div id="ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3,52px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#27ae60;text-transform:uppercase;margin-bottom:10px;">dil1(f,b)</p>
        <div id="ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3,52px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#c0392b;text-transform:uppercase;margin-bottom:10px;">ero1(f,b)</p>
        <div id="ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3,52px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0407Ready === '1') return;
    root.dataset.ep0407Ready = '1';

  var gB=root.querySelector('#ep0407_grid_b');
  var gF=root.querySelector('#ep0407_grid_f');
  var gD=root.querySelector('#ep0407_grid_d');
  var gE=root.querySelector('#ep0407_grid_e');
  if(!gB||!gF)return;
  var b=[[0,1,0],[1,2,1],[0,1,0]];
  var f=[[10,20,30],[40,50,60],[70,80,90]];
  var L=3,C=3;
  function compute(){
    var oy=-3/2+0.5, ox=-3/2+0.5;
    var dil=f.map(function(r){return r.slice();});
    var ero=f.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      for(var by=0;by<3;by++)for(var bx=0;bx<3;bx++){
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0&&vy<L&&vx>=0&&vx<C){
          var cd=f[vy][vx]+b[by][bx];
          if(cd>dil[y][x]) dil[y][x]=cd;
          var ce=f[vy][vx]-b[by][bx];
          if(ce<ero[y][x]) ero[y][x]=ce;
        }
      }
    }
    return {dil:dil,ero:ero};
  }
  function renderB(){
    gB.innerHTML='';
    for(var by=0;by<3;by++)for(var bx=0;bx<3;bx++){
      (function(by,bx){
        var wrap=document.createElement('div');
        wrap.style.cssText='display:flex;flex-direction:column;align-items:center;';
        var val=document.createElement('div');
        val.style.cssText='font-family:monospace;font-weight:bold;font-size:11px;color:#d35400;margin-bottom:2px;';
        val.innerText=b[by][bx];
        var sl=document.createElement('input');
        sl.type='range'; sl.min='-5'; sl.max='5'; sl.step='1'; sl.value=b[by][bx];
        sl.style.cssText='width:50px;accent-color:#d35400;';
        sl.oninput=function(){b[by][bx]=parseInt(sl.value); val.innerText=b[by][bx]; render_0407();};
        wrap.appendChild(val); wrap.appendChild(sl);
        gB.appendChild(wrap);
      })(by,bx);
    }
  }
  function paint(grid,img,color){
    grid.innerHTML='';
    var maxAbs=Math.max.apply(null,img.flat().map(Math.abs))||1;
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var v=img[y][x];
      var inten=Math.min(255,Math.max(0,v));
      c.style.cssText='width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      c.style.background='rgb('+inten+','+inten+','+inten+')';
      c.style.color=inten>128?'black':'white';
      c.innerText=v;
      grid.appendChild(c);
    }
  }
  function render_0407(){
    var res=compute();
    paint(gF,f); paint(gD,res.dil); paint(gE,res.ero);
  }
  renderB(); render_0407();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0407"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [62]:
import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

In [68]:
codigo = """
from morph import mm
L, C = int(input()), int(input())
Lb, Cb = int(input()), int(input())
b = mm.readImg(Lb, Cb)
f = mm.readImg(L, C)
r = mm.dil1(f, b)
print(mm.drawImg(r), end=" ")
r = mm.ero1(f, b)
print(mm.drawImg(r))
"""

TestSuite("EP04_07").run_code(codigo)

In [ ]:
%%writefile EP04_07.py
# Código Python
from morph import mm
L, C = int(input()),int(input())
Lb, Cb = int(input()),int(input())
b = mm.readImg(Lb, Cb)
f = mm.readImg(L, C)
r = mm.dil1(f, b)
print(mm.drawImg(r),end=" ")
r = mm.ero1(f, b)
print(mm.drawImg(r))

Overwriting EP04_07.py


In [60]:
TestSuite("EP04_07.py").run()

### EP04_08 🌋 Gradiente Morfológico, Top-hat e Black-hat

Em **inspeção automática de placas de circuito**, três perguntas aparecem o tempo todo: onde estão as **bordas** dos componentes? Quais **detalhes claros e pequenos** (como pontos de solda) se destacam do fundo? Quais **reentrâncias escuras** (como fissuras) o fundo esconde? Um único par erosão/dilatação responde às três: o **gradiente morfológico** evidencia contornos, o **top-hat** revela picos estreitos, e o **black-hat** revela vales estreitos — três ferramentas, uma só vizinhança.
Ver na @fig-04-sim-gradiente uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original, em tons de cinza), linha a linha.
5. **Operadores de base:** Calcular, exatamente como nos EPs 04_03 a 04_06:
   * $d = f \oplus B$ (dilatação),
   * $e = f \ominus B$ (erosão),
   * $\text{abertura} = e \oplus B$,
   * $\text{fechamento} = d \ominus B$.
6. **Gradiente morfológico:** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat:** $\text{tophat}(y,x) = f(y,x) - \text{abertura}(y,x)$.
8. **Black-hat:** $\text{blackhat}(y,x) = \text{fechamento}(y,x) - f(y,x)$.
9. **Saída:** Exibir, **nesta ordem**, as três matrizes completas: gradiente, top-hat, black-hat.

#### 📌 Restrições Computacionais

* **Sem padding em nenhuma etapa intermediária** — dilatação, erosão, abertura e fechamento seguem as mesmas regras de vizinhança dos EPs anteriores.
* **Não há *clipping*:** as três saídas podem conter qualquer valor inteiro (o gradiente é sempre $\geq 0$, mas top-hat e black-hat também).
* **Reaproveitamento:** $d$ e $e$ devem ser calculados **uma única vez** e reaproveitados para montar abertura, fechamento e gradiente.

#### 🧠 Fundamentação Teórica

| Operador | Fórmula | O que revela |
|----------|---------|----------------|
| **Gradiente** | $d - e$ | Bordas: zero em regiões planas, alto nas transições |
| **Top-hat** | $f - \text{abertura}(f)$ | Elementos **claros e finos**, menores que $B$ |
| **Black-hat** | $\text{fechamento}(f) - f$ | Elementos **escuros e finos**, menores que $B$ |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $L_B$.
* Linha 4: Inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.

**Saída:**

* Matriz gradiente em $L$ linhas e $C$ colunas.
* Matriz top-hat em $L$ linhas e $C$ colunas.
* Matriz black-hat em $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradiente: halo $3\times3=70$ em torno de $(2,2)$ e halo $3\times3=8$ em torno de $(6,6)$, resto $0$)<br>(top-hat: único $70$ em $(2,2)$, resto $0$)<br>(black-hat: único $8$ em $(6,6)$, resto $0$) | Pico isolado vira top-hat; vale isolado vira black-hat; ambos aparecem no gradiente |


In [40]:
#| label: fig-04-sim-gradiente
#| fig-cap: "Simulador: Gradiente Morfológico, Top-hat e Black-hat"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0408" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Gradiente / Top-hat / Black-hat</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🌋 3 operadores, 1 vizinhança</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="text-align:center;margin-bottom:14px;">
      <button id="ep0408_btn_pico" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #ccc;background:#fdf2e9;cursor:pointer;margin-right:6px;">☀️ Adicionar Pico</button>
      <button id="ep0408_btn_vale" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #ccc;background:#eaf2fb;cursor:pointer;margin-right:6px;">🕳️ Adicionar Vale</button>
      <button id="ep0408_btn_reset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Limpar</button>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">f</p>
        <div id="ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#6c3483;text-transform:uppercase;margin-bottom:8px;">gradiente</p>
        <div id="ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#d35400;text-transform:uppercase;margin-bottom:8px;">top-hat</p>
        <div id="ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#2980b9;text-transform:uppercase;margin-bottom:8px;">black-hat</p>
        <div id="ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
    </div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0408Ready === '1') return;
    root.dataset.ep0408Ready = '1';

  var gF=root.querySelector('#ep0408_grid_f');
  var gGrad=root.querySelector('#ep0408_grid_grad');
  var gTh=root.querySelector('#ep0408_grid_th');
  var gBh=root.querySelector('#ep0408_grid_bh');
  if(!gF)return;
  var L=9,C=9,f=[],B=[[1,1,1],[1,1,1],[1,1,1]];
  function reset_0408(){ f=Array.from({length:L},function(){return new Array(C).fill(10);}); }
  function morph(img,Bm,mode){
    var Bref=mode==='dil'?Bm.slice().reverse().map(function(r){return r.slice().reverse();}):Bm;
    var HB=Bref.length, WB=Bref[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bref[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C){
        if(mode==='dil' && img[vy][vx]>g[y][x]) g[y][x]=img[vy][vx];
        if(mode==='ero' && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
      }
    }
    return g;
  }
  function paint(grid,img,cmin,cmax,hue){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var v=img[y][x];
      var t=cmax>cmin?(v-cmin)/(cmax-cmin):0;
      var inten=Math.round(255-t*200);
      var c=document.createElement('div');
      c.style.cssText='width:20px;height:20px;border-radius:2px;';
      c.style.background= v===0 ? '#f2f2f2' : hue;
      c.style.opacity = v===0 ? '1' : (0.35+0.65*Math.min(1,t));
      grid.appendChild(c);
    }
  }
  function render_0408(){
    var d=morph(f,B,'dil');
    var e=morph(f,B,'ero');
    var ab=morph(e,B,'dil');
    var fc=morph(d,B,'ero');
    var grad=f.map(function(r,y){return r.map(function(_,x){return d[y][x]-e[y][x];});});
    var th=f.map(function(r,y){return r.map(function(v,x){return v-ab[y][x];});});
    var bh=f.map(function(r,y){return r.map(function(v,x){return fc[y][x]-v;});});
    var maxF=Math.max.apply(null,f.flat());
    paint(gF,f,10,maxF,'#7f8c8d');
    paint(gGrad,grad,0,Math.max(1,Math.max.apply(null,grad.flat())),'#6c3483');
    paint(gTh,th,0,Math.max(1,Math.max.apply(null,th.flat())),'#d35400');
    paint(gBh,bh,0,Math.max(1,Math.max.apply(null,bh.flat())),'#2980b9');
  }
  root.querySelector('#ep0408_btn_pico').onclick=function(){
    var y=2+Math.floor(Math.random()*5), x=2+Math.floor(Math.random()*5);
    f[y][x]=Math.min(255,f[y][x]+60+Math.floor(Math.random()*30));
    render_0408();
  };
  root.querySelector('#ep0408_btn_vale').onclick=function(){
    var y=2+Math.floor(Math.random()*5), x=2+Math.floor(Math.random()*5);
    f[y][x]=Math.max(0,f[y][x]-8-Math.floor(Math.random()*4));
    render_0408();
  };
  root.querySelector('#ep0408_btn_reset').onclick=function(){reset_0408();render_0408();};
  reset_0408();
  f[2][2]=80; f[6][6]=2;
  render_0408();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0408"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [41]:
%%writefile EP04_08.py
# Código Python

Writing EP04_08.py


In [42]:
TestSuite("EP04_08.py").run()

### EP04_09 🗺️ Transformada de Distância e o "Miolo" do Objeto

Em **robótica móvel**, ao planejar uma rota dentro de um corredor, o robô quer saber não apenas *onde* existe espaço livre, mas também **quão longe** cada ponto livre está da parede mais próxima. Os caminhos mais seguros tendem a passar pelo "miolo" do corredor, longe dos obstáculos.

A **transformada de distância morfológica** atribui a cada pixel um valor que representa sua distância até a borda mais próxima, segundo a métrica definida pelo elemento estruturante. Pixels próximos à borda recebem valores baixos, enquanto pixels mais internos recebem valores maiores. O pixel de valor máximo corresponde à região mais protegida do objeto, frequentemente associada ao seu centro morfológico.

Ver na @fig-04-sim-distancia uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** ler os inteiros $L$ (linhas) e $C$ (colunas) da imagem $f$.
2. **Dimensões de $B$:** ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** ler a matriz $b$, contendo valor $0$ no centro e valores negativos nas demais posições.
4. **Imagem:** ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Preparação:** multiplicar a imagem por $L\times C$, garantindo que os pixels internos tenham valor inicial suficientemente alto para a propagação das distâncias.
6. **Transformada de distância:** calcular a matriz de distâncias utilizando o método `mm.dist1(f,b)`.
7. **Saída:** exibir a matriz resultante da transformada de distância.

#### 📌 Restrições Computacionais

* Utilizar a implementação de erosão ponderada fornecida pela biblioteca.
* O elemento estruturante pode conter valores negativos arbitrários.
* A transformada deve ser obtida pela aplicação iterativa de erosões ponderadas até atingir um ponto fixo.

**⚠️ Nota Crucial sobre Leitura de Matrizes:** Como o elemento estruturante pode conter valores inteiros negativos (por exemplo, `-1` e `-99`), **não utilize a função `mm.readImg` para ler a matriz $b$**. Essa função converte os dados para o tipo `uint8`, provocando *underflow* e corrompendo os valores negativos. Leia as $L_B$ linhas de $b$ manualmente utilizando o tipo padrão `int`. A imagem $f$ pode continuar sendo lida normalmente por `mm.readImg`.

#### 🧠 Fundamentação Teórica

| Conceito                            | Significado                                                                       | Impacto Visual                               |
| ----------------------------------- | --------------------------------------------------------------------------------- | -------------------------------------------- |
| **$\text{dist}(y,x)$**              | Distância morfológica até a borda mais próxima segundo a métrica definida por $b$ | Pixels mais internos recebem valores maiores |
| **Valor máximo**                    | Pixel mais distante da borda                                                      | Aproxima o centro morfológico do objeto      |
| **Elemento estruturante ponderado** | Define os custos de deslocamento entre pixels vizinhos                            | Determina a métrica de distância utilizada   |
| **Objetos finos**                   | Regiões estreitas do objeto                                                       | Produzem valores baixos de distância         |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: inteiro $L$.
* Linha 2: inteiro $C$.
* Linha 3: inteiro $L_B$.
* Linha 4: inteiro $C_B$.
* Próximas $L_B$ linhas: elementos inteiros da matriz $b$.
* Próximas $L$ linhas: elementos binários ($0$ ou $1$) da matriz $f$.

⚠️ **Nota de implementação:** Os elementos da matriz $f$ (0 ou 1) devem ser multiplicados por **255** para gerar uma imagem binária adequada ($0$ e $255$) antes de aplicar a Transformada de Distância (TD).


**Saída:**

* Matriz da transformada de distância em $L$ linhas e $C$ colunas.

#### 📌 Exemplo

| Entrada                                                                                                                                                          | Saída                                                                                                 | Observação                              |
| ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- | --------------------------------------- |
| 5<br>9<br>3<br>3<br>-99 -1 -99<br>-1 0 -1<br>-99 -1 -99<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | Resultado da transformada de distância. |

**Nota:** o valor `-99` atua como uma aproximação prática de $-\infty$, impedindo a propagação pelas diagonais. Dessa forma, apenas os vizinhos horizontal e vertical contribuem para a distância, produzindo a distância de Manhattan.


In [43]:
#| label: fig-04-sim-distancia
#| fig-cap: "Simulador: Transformada de Distância"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0409" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Transformada de Distância</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🗺️ camadas de erosão</span>
  </div>

  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#555;margin-bottom:10px;text-align:center;">
      Clique nas células para desenhar seu próprio objeto (mínimo 3 pixels)
    </p>

    <div style="display:flex;justify-content:center;margin-bottom:18px;">
      <div id="ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9,32px);gap:2px;"></div>
    </div>

    <div style="text-align:center;margin-bottom:14px;">
      <button id="ep0409_btn_corredor" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;margin-right:6px;">📐 Corredor</button>
      <button id="ep0409_btn_disco" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;margin-right:6px;">⬤ Disco</button>
      <button id="ep0409_btn_l" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">📏 Forma em L</button>
    </div>

    <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;text-align:center;">
      Mapa de distâncias
    </p>

    <div style="display:flex;justify-content:center;">
      <div id="ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9,32px);gap:2px;"></div>
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.ep0409Ready === '1') return;
    root.dataset.ep0409Ready = '1';

    var gF=root.querySelector('#ep0409_grid_f');
    var gD=root.querySelector('#ep0409_grid_dist');

    var L=5,C=9,f=[];
    var B=[[0,1,0],[1,1,1],[0,1,0]];

    function setCorredor(){
      f=Array.from({length:L},()=>new Array(C).fill(0));
      for(var y=1;y<4;y++)for(var x=1;x<8;x++) f[y][x]=1;
    }

    function setDisco(){
      f=Array.from({length:L},()=>new Array(C).fill(0));
      var cy=2,cx=4;
      for(var y=0;y<L;y++)
        for(var x=0;x<C;x++)
          if(Math.pow(y-cy,2)+Math.pow((x-cx)*0.6,2)<=4) f[y][x]=1;
    }

    function setL(){
      f=Array.from({length:L},()=>new Array(C).fill(0));
      for(var y=1;y<4;y++)for(var x=1;x<3;x++) f[y][x]=1;
      for(var y=2;y<4;y++)for(var x=1;x<8;x++) f[y][x]=1;
    }

    function erode(img,Bm){
      var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
      var g=img.map(r=>r.slice());

      for(var y=0;y<L;y++)
        for(var x=0;x<C;x++)
          for(var by=0;by<HB;by++)
            for(var bx=0;bx<WB;bx++){
              if(Bm[by][bx]!==1) continue;
              var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
              if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]<g[y][x])
                g[y][x]=img[vy][vx];
            }
      return g;
    }

    function sameMatrix(a,b){
      for(var y=0;y<L;y++)
        for(var x=0;x<C;x++)
          if(a[y][x]!==b[y][x]) return false;
      return true;
    }

    function render(){
      gF.innerHTML='';
      for(var y=0;y<L;y++)for(var x=0;x<C;x++){
        (function(y,x){
          var c=document.createElement('div');
          c.style.cssText='width:32px;height:32px;border-radius:4px;border:1px solid #ddd;cursor:pointer;';
          c.style.background=f[y][x]?'#16a085':'#f7f7f7';
          c.onclick=function(){f[y][x]=1-f[y][x]; render();};
          gF.appendChild(c);
        })(y,x);
      }

      var dist=Array.from({length:L},()=>new Array(C).fill(0));
      var atual=f.map(r=>r.slice());
      var nivel=0;

      while(atual.some(r=>r.some(v=>v===1))){
        nivel++;
        for(var y=0;y<L;y++)
          for(var x=0;x<C;x++)
            if(atual[y][x]===1) dist[y][x]=nivel;

        var prox=erode(atual,B);
        if(sameMatrix(prox,atual)) break;
        atual=prox;
      }

      gD.innerHTML='';
      for(var y=0;y<L;y++)for(var x=0;x<C;x++){
        var c=document.createElement('div');
        var v=dist[y][x];
        var t=v/(nivel||1);
        var inten=Math.round(220-t*170);

        c.style.cssText='width:32px;height:32px;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:bold;border:1px solid #ddd;';
        c.style.background= v===0 ? '#f7f7f7' : 'rgb('+(inten-60)+','+inten+','+(inten-30)+')';
        c.style.color = v>0 ? 'white':'#bbb';
        c.innerText = v||'';
        gD.appendChild(c);
      }
    }

    root.querySelector('#ep0409_btn_corredor').onclick=()=>{setCorredor();render();};
    root.querySelector('#ep0409_btn_disco').onclick=()=>{setDisco();render();};
    root.querySelector('#ep0409_btn_l').onclick=()=>{setL();render();};

    setCorredor();
    render();
  }

  function tryInit(){
    var all=document.querySelectorAll('[id="sim-ep0409"]');
    if(!all.length) return false;
    init(all[all.length-1]);
    return true;
  }

  var t=0;
  var interval=setInterval(()=>{
    t++;
    if(tryInit() || t>20) clearInterval(interval);
  },50);
})();
</script>
""")

In [44]:
%%writefile EP04_09.py
# Código Python

Writing EP04_09.py


In [45]:
TestSuite("EP04_09.py").run()

### EP04_10 🪙 Separação de *Blobs*, Rotulação e Descritores

Em uma **linha de produção de moedas**, é comum que peças encostem umas nas outras na esteira, formando uma única mancha conectada na imagem — uma contagem ingênua erraria o total. A solução clássica combina operações morfológicas e análise de conectividade: primeiro uma **erosão** reduz ou rompe conexões frágeis entre objetos, e depois a **rotulação de componentes conectados** separa cada objeto em uma região distinta. Por fim, **descritores geométricos** (área e caixa delimitadora) resumem cada componente encontrado.

Ver na @fig-04-sim-rotulacao uma simulação deste EP.

---

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.

2. **Dimensões de $B$:** ler os inteiros $L_B$ (linhas) e $C_B$ (colunas) do elemento estruturante.

3. **Elemento estruturante:** ler a matriz $B$, contendo valores $0$ ou $1$, linha a linha.

4. **Dados:** ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.

5. **Separação:** calcular
   $$
   f_{ero} = f \ominus B
   $$
   usando erosão binária plana (como no EP04_04), eliminando conexões frágeis entre objetos.

6. **Rotulação:** sobre $f_{ero}$, identificar componentes conectados usando conectividade definida pela vizinhança $B$.  A rotulação deve seguir varredura *raster*: ao encontrar um pixel $1$ ainda não rotulado, atribuir um novo rótulo inteiro crescente a partir de 1 e propagar esse rótulo a toda a região conectada.

7. **Descritores:** para cada rótulo $k$, calcular:

   * **Área:** número de pixels pertencentes ao rótulo;
   * **Caixa delimitadora:** $$(y_{min}, x_{min}, y_{max}, x_{max})$$

8. **Saída:** exibir o número total de rótulos e, em seguida, uma linha por rótulo no formato:
   $$
   k,\ \text{área},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
   $$

---

#### 📌 Restrições Computacionais

* A erosão deve ser aplicada antes da rotulação.
* A conectividade é fixa e definida pela vizinhança acima.
* O elemento estruturante $B$ não interfere na conectividade da rotulação.
* Sem padding em qualquer etapa.
* A ordem dos rótulos segue a primeira descoberta em varredura *raster*.

---

#### 🧠 Fundamentação Teórica

| Conceito           | Significado                                 | Impacto                                              |
| ------------------ | ------------------------------------------- | ---------------------------------------------------- |
| Ponte fina         | Conexão estreita entre objetos              | Pode ser removida pela erosão morfológica            |
| Conectividade      | Definida pelo conjunto $$\mathcal{N}(y,x)$$ | Determina quais pixels pertencem ao mesmo componente |
| Área               | Número de pixels por componente             | Estimativa direta do tamanho do objeto               |
| Caixa delimitadora | Extensão espacial do rótulo                 | Resumo geométrico do componente                      |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: inteiro $L$
* Linha 2: inteiro $C$
* Linha 3: inteiro $L_B$
* Linha 4: inteiro $C_B$
* Próximas $L_B$ linhas: matriz $B$
* Próximas $L$ linhas: matriz $f$

**Saída:**

* Linha 1: número total de rótulos encontrados
* Linhas seguintes:
  $$
  k,\ \text{área},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
  $$


In [46]:
#| label: fig-04-sim-rotulacao
#| fig-cap: "Simulador: Separação de Blobs, Rotulação e Descritores"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0410" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Moedas Coladas → Separadas → Contadas</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🪙 erosão + rótulo + descritores</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:18px;text-align:center;">
      <label style="font-size:12px;font-weight:bold;color:#b9770e;">Espessura da ponte entre as moedas</label><br>
      <input id="ep0410_sl_p" style="width:60%;accent-color:#b9770e;margin-top:8px;" max="3" min="1" step="1" type="range" value="1">
      <span id="ep0410_vl_p" style="font-family:monospace;font-weight:bold;color:#b9770e;margin-left:8px;">1 px</span>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original (ligadas)</p>
        <div id="ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10,26px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">após erosão + rótulos</p>
        <div id="ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10,26px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0410_info" style="margin-top:18px;background:#fff8e1;border-radius:8px;padding:12px;border:1px solid #ffe0a3;font-size:12px;color:#7d5a00;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.ep0410Ready === '1') return;
    root.dataset.ep0410Ready = '1';

  var slP=root.querySelector('#ep0410_sl_p');
  var vlP=root.querySelector('#ep0410_vl_p');
  var gF=root.querySelector('#ep0410_grid_f');
  var gL=root.querySelector('#ep0410_grid_lab');
  var info=root.querySelector('#ep0410_info');
  if(!slP||!gF)return;
  var L=7,C=10;
  var B=[[1,1,1],[1,1,1],[1,1,1]];
  var palette=['#e74c3c','#27ae60','#2980b9','#8e44ad','#d35400'];
  function buildF(p){
    var f=Array.from({length:L},function(){return new Array(C).fill(0);});
    for(var y=1;y<6;y++)for(var x=1;x<4;x++) f[y][x]=1;
    for(var y=1;y<6;y++)for(var x=6;x<9;x++) f[y][x]=1;
    var midRow=3;
    for(var dy=0;dy<p;dy++){
      var ry=midRow-Math.floor(p/2)+dy;
      for(var x=4;x<6;x++) f[ry][x]=1;
    }
    return f;
  }
  function erode(img,Bm){
    var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bm[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function labelK8(img){
    var labels=Array.from({length:L},function(){return new Array(C).fill(0);});
    var dirs=[[-1,-1],[-1,0],[-1,1],[0,-1],[0,1],[1,-1],[1,0],[1,1]];
    var cur=0, desc=[];
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      if(img[y][x]===1 && labels[y][x]===0){
        cur++;
        var stack=[[y,x]];
        labels[y][x]=cur;
        var area=0, miny=y,maxy=y,minx=x,maxx=x;
        while(stack.length){
          var cell=stack.pop(); var cy=cell[0],cx=cell[1];
          area++;
          if(cy<miny)miny=cy; if(cy>maxy)maxy=cy;
          if(cx<minx)minx=cx; if(cx>maxx)maxx=cx;
          dirs.forEach(function(d){
            var ny=cy+d[0],nx=cx+d[1];
            if(ny>=0&&ny<L&&nx>=0&&nx<C&&img[ny][nx]===1&&labels[ny][nx]===0){
              labels[ny][nx]=cur; stack.push([ny,nx]);
            }
          });
        }
        desc.push({k:cur,area:area,miny:miny,minx:minx,maxy:maxy,maxx:maxx});
      }
    }
    return {labels:labels, desc:desc};
  }
  function paintBin(grid,img){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ddd;';
      c.style.background=img[y][x]?'#b9770e':'#fdf6ec';
      grid.appendChild(c);
    }
  }
  function paintLabels(grid,labels){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var k=labels[y][x];
      c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ddd;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;color:white;';
      c.style.background = k>0 ? palette[(k-1)%palette.length] : '#fdf6ec';
      c.innerText = k>0 ? k : '';
      grid.appendChild(c);
    }
  }
  function render_0410(){
    var p=parseInt(slP.value);
    vlP.innerText=p+' px';
    var f=buildF(p);
    var fe=erode(f,B);
    var res=labelK8(fe);
    paintBin(gF,f);
    paintLabels(gL,res.labels);
    var txt='<b>'+res.desc.length+' objeto(s) detectado(s) após a erosão.</b><br>';
    res.desc.forEach(function(d){
      txt+='Rótulo '+d.k+': área='+d.area+', bbox=('+d.miny+','+d.minx+')→('+d.maxy+','+d.maxx+')<br>';
    });
    if(res.desc.length<2){
      txt+='<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
    }
    info.innerHTML=txt;
  }
  slP.oninput=render_0410;
  render_0410();
  }

  function tryInit(){
    var all = document.querySelectorAll('[id="sim-ep0410"]');
    if(all.length === 0) return false;
    init(all[all.length-1]);
    return true;
  }

  var tentativas = 0;
  var intervalo = setInterval(function(){
    tentativas++;
    if(tryInit() || tentativas > 20) clearInterval(intervalo);
  }, 50);
})();
</script>
""")


In [47]:
%%writefile EP04_10.py
# Código Python

Writing EP04_10.py


In [48]:
TestSuite("EP04_10.py").run()